# Тема 9. Типы взаимодействия между агентами

**Модуль III · Пример 2 из 5**

Три базовых типа межагентного взаимодействия и когда какой уместен:

| Тип | Идея | Когда применять | Цена |
|-----|------|-----------------|------|
| **Коллаборация** (handoff) | координатор делит задачу и делегирует специалистам, затем агрегирует | чёткий пайплайн, разные навыки | умеренная |
| **Дебаты** | агенты в несколько раундов критикуют решения друг друга | нужна точность, фактчекинг, оценка рисков | выше (раунды) |
| **Конкуренция** | несколько стратегий решают задачу независимо, оценщик выбирает лучшую | креативные/многовариантные задачи | самая высокая |

Реализуем по одному минимальному примеру каждого типа на домене «ШопБот».

In [1]:
import os, json
from openai import OpenAI

BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:8000/v1")
API_KEY  = os.environ.get("LLM_API_KEY", "")
MODEL    = os.environ.get("LLM_MODEL", "qwen/qwen3.7-max")
client = OpenAI(api_key=API_KEY or "EMPTY", base_url=BASE_URL)

def chat(system: str, user: str, max_tokens: int = 400) -> str:
    """Короткий помощник: один запрос system+user, вернуть текст ответа."""
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return (r.choices[0].message.content or "").strip()

print("Готово:", MODEL)

Готово: qwen/qwen3.7-max


## 1. Коллаборация (handoff)

Координатор декомпозирует запрос на подзадачи и **передаёт** каждую своему специалисту (handoff), затем собирает результаты. Здесь — упрощённый пайплайн из двух специалистов (по политикам и по доставке); полная иерархия с планировщиком и критиком — в примере 3.

In [2]:
def policy_specialist(question: str) -> str:
    return chat("Ты специалист по политикам магазина. Отвечай кратко по существу вопроса о "
                "возвратах/доставке/гарантии.", question, 200)

def shipping_specialist(question: str) -> str:
    return chat("Ты специалист по срокам и вариантам доставки. Отвечай кратко.", question, 200)

def collaboration(user_query: str) -> str:
    # Координатор делит задачу (здесь — простая фиксированная декомпозиция для наглядности)
    print("Координатор: делю запрос на 2 подзадачи и делегирую специалистам (handoff)")
    r_policy = policy_specialist("Правила возврата товара?")
    print("  -> policy_specialist:", r_policy[:90])
    r_ship = shipping_specialist("Сроки стандартной и экспресс-доставки?")
    print("  -> shipping_specialist:", r_ship[:90])
    # Координатор агрегирует
    return chat("Ты координатор. Собери из ответов специалистов единый ответ пользователю.",
                f"Запрос: {user_query}\n\nОтвет по политикам: {r_policy}\n\nОтвет по доставке: {r_ship}")

ans = collaboration("Хочу вернуть товар и заказать новый — какие правила возврата и сроки доставки?")
print("\n=== Итог (координатор) ===\n", ans[:500])

Координатор: делю запрос на 2 подзадачи и делегирую специалистам (handoff)


  -> policy_specialist: **Правила возврата:**

* **Сроки:** До 14 дней для товаров надлежащего качества; в течение


  -> shipping_specialist: В среднем по рынку:
• **Стандартная:** 3–7 рабочих дней.
• **Экспресс:** 1–2 рабочих дня (



=== Итог (координатор) ===
 Здравствуйте! С радостью помогу вам разобраться с возвратом и расскажу, как быстро вы сможете получить новый заказ. 

Поскольку возврат средств занимает немного времени, **новый заказ лучше оформить отдельной покупкой**, чтобы не ждать поступления денег на счет.

Ниже собрал для вас всю необходимую информацию:

### 📦 Как оформить возврат
1. Создайте заявку на возврат в **личном кабинете** или обратитесь в нашу **службу поддержки**.
2. При себе нужно иметь чек, номер заказа или любое другое подтв


## 2. Дебаты

Два агента дают независимые оценки одной задачи, затем в несколько раундов **критикуют** аргументы друг друга и сходятся к решению. Повышает точность ценой дополнительных вызовов. Сценарий: спорный кейс — одобрять ли возврат.

In [3]:
CASE = ("Клиент просит возврат за наушники (заказ доставлен 20 дней назад). "
        "Политика: возврат в течение 14 дней с момента доставки. "
        "Клиент утверждает, что товар бракованный.")

def debate(rounds: int = 2) -> str:
    pos_a = chat("Ты агент-адвокат клиента. Аргументируй ЗА возврат, кратко.", CASE, 150)
    pos_b = chat("Ты агент-контролёр политик. Аргументируй ПРОТИВ или условия, кратко.", CASE, 150)
    print("Раунд 0:\n  A (за):", pos_a[:110], "\n  B (против):", pos_b[:110])
    for k in range(rounds):
        pos_a = chat("Ты адвокат клиента. Учти контраргумент оппонента и уточни позицию, кратко.",
                     f"Кейс: {CASE}\nОппонент: {pos_b}\nТвоя прежняя позиция: {pos_a}", 150)
        pos_b = chat("Ты контролёр политик. Учти довод оппонента и уточни позицию, кратко.",
                     f"Кейс: {CASE}\nОппонент: {pos_a}\nТвоя прежняя позиция: {pos_b}", 150)
        print(f"Раунд {k+1}:\n  A:", pos_a[:110], "\n  B:", pos_b[:110])
    return chat("Ты арбитр. На основе дебатов вынеси взвешенное итоговое решение по кейсу.",
                f"Кейс: {CASE}\nПозиция A: {pos_a}\nПозиция B: {pos_b}")

print("\n=== Решение арбитра ===\n", debate()[:500])

Раунд 0:
  A (за): Как представитель клиента, требую оформить возврат на основании следующих аргументов:

1. **Разграничение осно 
  B (против): **ПРОТИВ (стандартного возврата):** 
Заявка на стандартный возврат отклонена, так как нарушены сроки политики:


Раунд 1:
  A: **Уточненная позиция (ответ на контраргументы оппонента):**

Оппонент пытается подменить понятия, смешивая пра 
  B: **Довод принят:** Разграничение ст. 25 (14 дней для качественного товара) и ст. 18 ЗоЗПП (брак) корректно. Ссы


Раунд 2:
  A: **Уточненная позиция (завершение и ответ на доводы оппонента):**

...действует гарантийный срок). Оппонент кор 
  B: **Довод оппонента учтен частично.** 
Оппонент прав в части императивного срока (ст. 22 ЗоЗПП) и бремени доказы



=== Решение арбитра ===
 Как арбитр, рассмотрев доводы обеих сторон и опираясь на нормы Закона РФ «О защите прав потребителей» (ЗоЗПП) и сложившуюся судебную практику, я выношу следующее взвешенное решение.

### Мотивировочная часть (Анализ доводов)

1. **Применимость политики возврата (14 дней vs Гарантия):**
   Внутренняя политика компании о 14 днях применима только к возврату товара *надлежащего качества* (ст. 25 ЗоЗПП). Поскольку клиент заявил о *браке*, дело автоматически переходит в плоскость ст. 18 ЗоЗПП. Срок в 


## 3. Конкуренция

Несколько агентов предлагают **разные стратегии** решения, независимый **оценщик** выбирает лучшую по заданным критериям. Сценарий: рекомендация варианта доставки под приоритет клиента.

In [4]:
REQUEST = "Клиенту нужна посылка к важной дате через 3 дня, бюджет ограничен. Что порекомендовать?"

def competition() -> str:
    strategies = {
        "эконом":  chat("Ты стратег 'дешевле'. Предложи вариант доставки с упором на цену.", REQUEST, 120),
        "быстро":  chat("Ты стратег 'быстрее'. Предложи вариант с упором на срок.", REQUEST, 120),
        "баланс":  chat("Ты стратег 'баланс цена/срок'. Предложи компромисс.", REQUEST, 120),
    }
    for name, s in strategies.items():
        print(f"  [{name}] {s[:100]}")
    joined = "\n".join(f"[{n}] {s}" for n, s in strategies.items())
    return chat("Ты оценщик. Выбери ЛУЧШУЮ стратегию под запрос (срок 3 дня + ограниченный бюджет) "
                "и обоснуй выбор одним абзацем.", f"Запрос: {REQUEST}\n\nВарианты:\n{joined}")

print("\n=== Выбор оценщика ===\n", competition()[:500])

  [эконом] Привет. Я стратег «Дешевле». 

Сразу снимем розовые очки: в логистике есть железное правило **«Быстр
  [быстро] Слушай внимательно. Времени на раскачку нет. 3 дня — это уже не срок, это обратный отсчет. Конфликт 
  [баланс] Здравствуйте. Как стратег баланса, я вижу классический конфликт: **срок (3 дня) требует премиум-тари



=== Выбор оценщика ===
 Лучшей стратегией является **[баланс]**, поскольку она предлагает наиболее профессиональный и безопасный выход из классического логистического треугольника: вместо рискованной жертвы надежностью (как в варианте «эконом», что категорически недопустимо для «важной даты») или излишне агрессивного тона (как в «быстро»), она честно предлагает клиенту компенсировать нехватку бюджета собственным комфортом и временем (например, отказом от сервиса «от двери до двери» в пользу самостоятельного отвоза на т


## Итоги

- **Коллаборация** — рабочая лошадка МАС: декомпозиция + handoff + агрегация; предсказуема и понятна.
- **Дебаты** повышают точность в спорных/рисковых кейсах ценой дополнительных раундов (вызовов модели).
- **Конкуренция** хороша, когда полезно перебрать разные стратегии, но она самая ресурсоёмкая.
- Тип взаимодействия выбирается **под задачу**; в иерархической МАС (пример 3) основой служит коллаборация, а элементы дебатов/конкуренции подключаются точечно (например, критик — это верификация, близкая к дебатам).

**Дальше:** Тема 10 — архитектуры МАС: иерархия с планировщиком, специалистами, координатором и критиком.